<h1 align='center'> 영상처리 프로그래밍 실습 7</h1>

<h6 align='right'> 2025. 5. 8. </h6>

<div class="alert alert-block alert-info">
    
- 파일 이름에서 00000000을 자신의 학번으로, name을 자신의 이름으로 수정하세요.

- 다음 줄에 자신의 이름, 학번, 학과(전공)을 적으세요.

* 이름:   이한석         학번:    20215226        학과(전공): 빅데이터학과
    
</div>

- JupyterLab 문서의 최신 버전은 [JupyterLab Documentation](https://jupyterlab.readthedocs.io/en/stable/index.html#/)을  참고하라

- Markdown은 [Markdown Guide](https://www.markdownguide.org/)를 참고하라.
- [Markdown Cheat Sheet](https://www.markdownguide.org/cheat-sheet/)

* 제출 마감: 5월 14일 (수) 오후 10:00까지 최종본을 SmartLEAD제출


In [1]:
import cv2
import matplotlib.pyplot as plt

import numpy as np
print("OpenCV version", cv2.__version__)
print("NumPy version", np.__version__)

OpenCV version 4.11.0
NumPy version 2.0.2


## 문제 1.

지난 주에 작성했던 PyQt5를 이용한 GUI에 몇 가지 기능을 추가했던 프로그램을 실행한 후에 윈도우의 크기를 조정해 보자. 윈도우의 크기를 조정한 후에 영상의 크기도 윈도우의 크기에 비례해서 조정하려면 어떤 기능을 추가해야 하는지 조사해서 다시 작성하라.

In [ ]:
import sys
from PyQt5.QtWidgets import (
    QApplication, 
    QMainWindow, 
    QLabel,
    QMenuBar, QAction,
    QFileDialog, QHBoxLayout, QWidget,
    QSizePolicy
)
from PyQt5.QtCore import Qt 
from PyQt5.QtGui import QPixmap, QImage
import image_tools  # 사용자 정의 모듈
import numpy as np

class BasicViewer(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("단순 이미지 뷰어")
        self.setGeometry(100, 100, 1000, 600)

        # QLabel 두 개: 원본 / 변환
        self.original_label = QLabel("원본 이미지를 열어보세요")
        self.original_label.setAlignment(Qt.AlignCenter)
        self.original_label.setSizePolicy(QSizePolicy.Ignored, QSizePolicy.Ignored)

        self.processed_label = QLabel("변환된 이미지가 여기에 표시됩니다")
        self.processed_label.setAlignment(Qt.AlignCenter)
        self.processed_label.setSizePolicy(QSizePolicy.Ignored, QSizePolicy.Ignored)

        # 수평 레이아웃
        self.central_widget = QWidget()
        self.hbox = QHBoxLayout(self.central_widget)
        self.hbox.addWidget(self.original_label)
        self.hbox.addWidget(self.processed_label)

        self.setCentralWidget(self.central_widget)

        self.current_image = None  # 원본 이미지 (NumPy)
        self.transformed_image = None  # 변환 이미지

        self.create_menu()

    def create_menu(self):
        menubar = self.menuBar()

        # 파일 메뉴
        file_menu = menubar.addMenu("파일")
        open_action = QAction("열기", self)
        open_action.triggered.connect(self.open_image_dialog)
        file_menu.addAction(open_action)

        exit_action = QAction("종료", self)
        exit_action.triggered.connect(self.close)
        file_menu.addAction(exit_action)

        # 변환 메뉴
        transform_menu = menubar.addMenu("변환")
        log_action = QAction("로그 변환", self)
        log_action.triggered.connect(self.apply_log_transform)
        transform_menu.addAction(log_action)

    def open_image_dialog(self):
        filename, _ = QFileDialog.getOpenFileName(
            self, "이미지 열기", "", "Image Files (*.png *.jpg *.bmp *.jpeg *.tif *.tiff)"
        )
        if filename:
            image = image_tools.load_image(filename)  # 흑백 또는 RGB ndarray
            self.current_image = image
        


#### 수정 후 프로그램
- 원 영상과 처리된 영상을 표시하는 두 개의 QLabel의 setSizePolicy 함수에 QSizePolicy.Ignored 인자를 전달
- resizeEvent 처리 함수를 추가하여 윈도우 크기가 변할 때마다 다시 영상의 크기를 조정하여 표시

In [11]:
import sys
from PyQt5.QtWidgets import (
    QApplication, 
    QMainWindow, 
    QLabel,
    QMenuBar, QAction,
    QFileDialog, QHBoxLayout, QWidget,
    QSizePolicy
)
from PyQt5.QtCore import Qt 
from PyQt5.QtGui import QPixmap, QImage
import image_tools  # 사용자 정의 모듈
import numpy as np

class BasicViewer(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("단순 이미지 뷰어")
        self.setGeometry(100, 100, 1000, 600)

        # QLabel 두 개: 원본 / 변환
        self.original_label = QLabel("원본 이미지를 열어보세요")
        self.original_label.setAlignment(Qt.AlignCenter)
        self.original_label.setSizePolicy(QSizePolicy.Ignored, QSizePolicy.Ignored)

        self.processed_label = QLabel("변환된 이미지가 여기에 표시됩니다")
        self.processed_label.setAlignment(Qt.AlignCenter)
        self.processed_label.setSizePolicy(QSizePolicy.Ignored, QSizePolicy.Ignored)

        # 수평 레이아웃
        self.central_widget = QWidget()
        self.hbox = QHBoxLayout(self.central_widget)
        self.hbox.addWidget(self.original_label)
        self.hbox.addWidget(self.processed_label)

        self.setCentralWidget(self.central_widget)

        self.current_image = None  # 원본 이미지 (NumPy)
        self.transformed_image = None  # 변환 이미지

        self.create_menu()

    def create_menu(self):
        menubar = self.menuBar()

        # 파일 메뉴
        file_menu = menubar.addMenu("파일")
        open_action = QAction("열기", self)
        open_action.triggered.connect(self.open_image_dialog)
        file_menu.addAction(open_action)

        exit_action = QAction("종료", self)
        exit_action.triggered.connect(self.close)
        file_menu.addAction(exit_action)

        # 변환 메뉴
        transform_menu = menubar.addMenu("변환")
        log_action = QAction("로그 변환", self)
        log_action.triggered.connect(self.apply_log_transform)
        transform_menu.addAction(log_action)

    def open_image_dialog(self):
        filename, _ = QFileDialog.getOpenFileName(
            self, "이미지 열기", "", "Image Files (*.png *.jpg *.bmp *.jpeg *.tif *.tiff)"
        )
        if filename:
            image = image_tools.load_image(filename)  # 흑백 또는 RGB ndarray
            self.current_image = image
            self.transformed_image = None
            self.update_display_images()

    def apply_log_transform(self):
        if self.current_image is None:
            return

        image = self.current_image.astype(np.float32) #로그 변환 공식
        C = 255 / np.log(1 + np.max(image)) #정규화를 위한 상수 C
        log_image = C * np.log(1 + image)
        
        log_image = np.clip(log_image, 0, 255).astype(np.uint8) #값 범위 0~255로 클리핑 후 정수형으로 변환

        self.transformed_image = log_image
        self.update_display_images()

    def update_display_images(self):
        if self.current_image is not None:
            orig_pixmap = self.numpy_to_qpixmap(self.current_image)
            self.original_label.setPixmap(orig_pixmap.scaled(
                self.original_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation
            ))

        if self.transformed_image is not None:
            proc_pixmap = self.numpy_to_qpixmap(self.transformed_image)
            self.processed_label.setPixmap(proc_pixmap.scaled(
                self.processed_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation
            ))
        else:
            self.processed_label.clear()
        
    def update_image_views(self):
        if self.current_image is not None:
            pixmap = self.numpy_to_qpixmap(self.current_image)
            scaled = pixmap.scaled(
                self.original_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation
            )
            self.original_label.setPixmap(scaled)
            
        if self.processed_image is not None:
            pixmap = self.numpy_to_qpixmap(self.processed_image)
            scaled = pixmap.scaled(
                self.processed_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation
            )
            self.processed_label.setPixmap(scaled)

    def numpy_to_qpixmap(self, image):
        if image.ndim == 2:  # 흑백이면 RGB로 변환
            image = np.stack([image] * 3, axis=-1)

        h, w, ch = image.shape
        bytes_per_line = ch * w
        qimage = QImage(image.data, w, h, bytes_per_line, QImage.Format_RGB888)
        return QPixmap.fromImage(qimage)

# 애플리케이션 실행
app = QApplication.instance()
if app is None:
    app = QApplication([])

viewer = BasicViewer()
viewer.show()

try:
    sys.exit(app.exec_())
except SystemExit:
    pass



### 문제 2. 

다음을 만족하는 함수 moving_average_filtering(filename)를 완성하라.

- filename으로 전달받은 파일이 존재하는지 조사해서 존재하지 않으면 오류 메시지를 콘솔에 출력하고 return 한다.
- 파일이 존재하면
  - 파일을 읽고, 원본 영상과 이동 평균 필터링된 영상을 나란히 화면에 표시한다.
  - 이동 평균 필터의 커널의 크기를 지정할 수 있는 trackBar interface를 제공한다. 단, trackBar로 지정하는 값의 범위는 1부터 51까지로 하되, 짝수일 경우 그보다 1 작은 홀수로 커널의 크기를 지정한다.
  - 지정된 커널의 크기를 이용해서 원 영상에 이동 평균 필터링이 적용된 영상을 구한다.

### 2.1 OpenCV GUI 기반

In [ ]:
import cv2 #OpenCV 기반
import numpy as np
import os

def moving_average_filtering_cv(filename):
    # 파일 확인
    if not os.path.exists(filename):
        print(f"[오류] 파일 '{filename}'이 존재하지 않습니다.")
        return

    #이미지 불러오기
    img = cv2.imread(filename)
    if img is None:
        print(f"[오류] 이미지를 불러올 수 없습니다: {filename}")
        return

    window_name = "OpenCV Moving Average"
    cv2.namedWindow(window_name)

    def on_trackbar(val): 
        kernel = max(1, val) #최소 커널 크기 1
        if kernel % 2 == 0: #커널 크기는 홀수
            kernel -= 1
        filtered = cv2.blur(img, (kernel, kernel)) #평균 블러 적용
        combined = np.hstack((img, filtered)) #원본+필터링 결과 나란히 표시
        cv2.imshow(window_name, combined) #화면에 표시

    #트랙바 생성
    cv2.createTrackbar("Kernel", window_name, 1, 51, on_trackbar)
    
    on_trackbar(1) 
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    
moving_average_filtering_cv("bird.jpg")


### 2.2 PyQt 기반

In [22]:
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QLabel, QSlider,
    QWidget, QHBoxLayout, QVBoxLayout, QSizePolicy
)
from PyQt5.QtCore import Qt
from PyQt5.QtGui import QPixmap, QImage, QFont
import numpy as np
import cv2
import os
import sys

class MovingAverageViewer(QMainWindow):
    def __init__(self, filename):
        super().__init__()
        self.setWindowTitle("PyQt 이동 평균 필터링")
        self.setGeometry(200, 200, 1200, 600)

        #파일이 존재하는지 확인
        if not os.path.exists(filename):
            print(f"[오류] 파일 '{filename}'이 존재하지 않습니다.")
            return

        #이미지 읽어오기
        self.original_image = cv2.imread(filename)
        if self.original_image is None:
            print(f"[오류] 이미지를 불러올 수 없습니다: {filename}")
            return

        self.filtered_image = None

        # 이미지 표시용 QLabel
        self.orig_label = QLabel()
        self.orig_label.setSizePolicy(QSizePolicy.Ignored, QSizePolicy.Ignored)
        self.orig_label.setAlignment(Qt.AlignCenter)

        #필터링 이미지 표시용 
        self.filtered_label = QLabel()
        self.filtered_label.setSizePolicy(QSizePolicy.Ignored, QSizePolicy.Ignored)
        self.filtered_label.setAlignment(Qt.AlignCenter)

        # 커널 사이즈 표시용 텍스트
        self.kernel_label = QLabel("Kernel Size: 3")
        self.kernel_label.setFont(QFont("Arial", 12))

        # 슬라이더
        self.slider = QSlider(Qt.Horizontal)
        self.slider.setMinimum(1)
        self.slider.setMaximum(51)
        self.slider.setValue(1)
        self.slider.valueChanged.connect(self.update_filter)

        # 슬라이더와 텍스트를 옆으로 배치
        slider_layout = QHBoxLayout()
        slider_layout.addWidget(QLabel("Kernel:"))
        slider_layout.addWidget(self.slider)
        slider_layout.addWidget(self.kernel_label)

        # 이미지 나란히 배치
        image_layout = QHBoxLayout()
        image_layout.addWidget(self.orig_label)
        image_layout.addWidget(self.filtered_label)

        # 전체 레이아웃
        main_layout = QVBoxLayout()
        main_layout.addLayout(image_layout)
        main_layout.addLayout(slider_layout)

        central_widget = QWidget()
        central_widget.setLayout(main_layout)
        self.setCentralWidget(central_widget)

        self.update_filter(3)

    #슬라이더 값이 바뀔 떄 필터 적용
    def update_filter(self, val):
        kernel = max(1, val)
        if kernel % 2 == 0:
            kernel -= 1 #커널은 홀수

        self.kernel_label.setText(f"{kernel}") #커널 크기 표시

        blurred = cv2.blur(self.original_image, (kernel, kernel))
        self.filtered_image = blurred

        self.orig_label.setPixmap(self.cv2_to_pixmap(self.original_image).scaled(
            self.orig_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))
        self.filtered_label.setPixmap(self.cv2_to_pixmap(blurred).scaled(
            self.filtered_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))

    #창 크기 변경 시 이미지 다시 스케일링
    def resizeEvent(self, event):
        super().resizeEvent(event)
        self.update_filter(self.slider.value())

    #OpenCV 이미지를 QPixmap으로 변환
    def cv2_to_pixmap(self, img):
        rgb_image = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, ch = rgb_image.shape
        bytes_per_line = ch * w
        qimage = QImage(rgb_image.data, w, h, bytes_per_line, QImage.Format_RGB888)
        return QPixmap.fromImage(qimage)

# 실행 함수
def moving_average_filtering_pyqt(filename):
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    viewer = MovingAverageViewer(filename)
    viewer.show()
    app.exec_()


moving_average_filtering_pyqt("bird.jpg")


## 문제 3.
강의에서 설명한 여러 가지 필터를 필터 메뉴에 추가하라.

In [ ]:
from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QLabel, QSlider,
    QWidget, QHBoxLayout, QVBoxLayout, QSizePolicy, QMenuBar, QAction
)
from PyQt5.QtCore import Qt
from PyQt5.QtGui import QPixmap, QImage, QFont
import numpy as np
import cv2
import os
import sys

class MovingAverageViewer(QMainWindow):
    def __init__(self, filename):
        super().__init__()
        self.setWindowTitle("PyQt 이미지 필터링")
        self.setGeometry(200, 200, 1200, 600)

        # 파일이 존재하는지 확인
        if not os.path.exists(filename):
            print(f"[오류] 파일 '{filename}'이 존재하지 않습니다.")
            return

        # 이미지를 읽어오기
        self.original_image = cv2.imread(filename)
        if self.original_image is None:
            print(f"[오류] 이미지를 불러올 수 없습니다: {filename}")
            return

        self.filtered_image = None

        # 이미지 표시용 QLabel
        self.orig_label = QLabel()
        self.orig_label.setSizePolicy(QSizePolicy.Ignored, QSizePolicy.Ignored)
        self.orig_label.setAlignment(Qt.AlignCenter)

        # 필터링 이미지 표시용
        self.filtered_label = QLabel()
        self.filtered_label.setSizePolicy(QSizePolicy.Ignored, QSizePolicy.Ignored)
        self.filtered_label.setAlignment(Qt.AlignCenter)

        # 커널 사이즈 표시용 텍스트
        self.kernel_label = QLabel("Kernel Size: 3")
        self.kernel_label.setFont(QFont("Arial", 12))

        # 슬라이더
        self.slider = QSlider(Qt.Horizontal)
        self.slider.setMinimum(1)
        self.slider.setMaximum(51)
        self.slider.setValue(3)
        self.slider.valueChanged.connect(self.apply_filter)

        # 메뉴 바 생성
        self.create_menu()

        # 레이아웃 설정
        slider_layout = QHBoxLayout()
        slider_layout.addWidget(QLabel("Kernel:"))
        slider_layout.addWidget(self.slider)
        slider_layout.addWidget(self.kernel_label)

        image_layout = QHBoxLayout()
        image_layout.addWidget(self.orig_label)
        image_layout.addWidget(self.filtered_label)

        main_layout = QVBoxLayout()
        main_layout.addLayout(image_layout)
        main_layout.addLayout(slider_layout)

        central_widget = QWidget()
        central_widget.setLayout(main_layout)
        self.setCentralWidget(central_widget)

        self.apply_filter(self.slider.value())  # 필터 적용 초기화

    #필터 메뉴 만들기
    def create_menu(self):
        menubar = self.menuBar()

        # 필터 메뉴
        filter_menu = menubar.addMenu("필터")

        # 이동 평균 필터
        mean_filter_action = QAction("이동 평균 필터", self)
        mean_filter_action.triggered.connect(lambda: self.apply_filter("mean"))
        filter_menu.addAction(mean_filter_action)

        # 가우시안 필터
        gaussian_filter_action = QAction("가우시안 필터", self)
        gaussian_filter_action.triggered.connect(lambda: self.apply_filter("gaussian"))
        filter_menu.addAction(gaussian_filter_action)

        # 미디언 필터
        median_filter_action = QAction("미디언 필터", self)
        median_filter_action.triggered.connect(lambda: self.apply_filter("median"))
        filter_menu.addAction(median_filter_action)

        # 박스 필터
        box_filter_action = QAction("박스 필터", self)
        box_filter_action.triggered.connect(lambda: self.apply_filter("box"))
        filter_menu.addAction(box_filter_action)

    # 필터 적용
    def apply_filter(self, val):
        kernel = self.slider.value()
        if kernel % 2 == 0:
            kernel -= 1  # 커널은 홀수

        self.kernel_label.setText(f"{kernel}")  # 커널 크기 표시

        # 필터 선택
        if isinstance(val, int):  # 슬라이더 값일 경우
            filter_type = "mean"  # 기본값은 이동 평균 필터로 설정
        else:
            filter_type = val

        if filter_type == "mean":
            # 이동 평균 필터
            filtered = cv2.blur(self.original_image, (kernel, kernel))
        elif filter_type == "gaussian":
            # 가우시안 필터
            filtered = cv2.GaussianBlur(self.original_image, (kernel, kernel), 0)
        elif filter_type == "median":
            # 미디언 필터
            filtered = cv2.medianBlur(self.original_image, kernel)
        elif filter_type == "box":
            # 박스 필터
            filtered = cv2.boxFilter(self.original_image, -1, (kernel, kernel))

        self.filtered_image = filtered

        # 원본 이미지와 필터링된 이미지 표시
        self.orig_label.setPixmap(self.cv2_to_pixmap(self.original_image).scaled(
            self.orig_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))
        self.filtered_label.setPixmap(self.cv2_to_pixmap(filtered).scaled(
            self.filtered_label.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation))

    # 창 크기 변경 시 이미지 다시 스케일링
    def resizeEvent(self, event):
        super().resizeEvent(event)
        self.apply_filter(self.slider.value())

    # OpenCV 이미지를 QPixmap으로 변환
    def cv2_to_pixmap(self, img):
        rgb_image = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w, ch = rgb_image.shape
        bytes_per_line = ch * w
        qimage = QImage(rgb_image.data, w, h, bytes_per_line, QImage.Format_RGB888)
        return QPixmap.fromImage(qimage)

# 실행 함수
def moving_average_filtering_pyqt(filename):
    app = QApplication.instance()
    if app is None:
        app = QApplication(sys.argv)
    viewer = MovingAverageViewer(filename)
    viewer.show()
    app.exec_()

# 호출
moving_average_filtering_pyqt("bird.jpg")


### 문제 4.

다음 URL에서 'Lincoln.jpg',  'Lincoln_wave_1.png', 'Lincoln_wave_2.png' 파일을 로컬 드라이브에 다운로드하라.

https://drive.google.com/drive/u/0/folders/1zbjtkf9nHy9VniuLI4wHilbrN_JBvhYi

로컬 드라이브에서 'Lincoln.jpg' 파일을 읽고, 'Lincoln_wave_1.png' 또는 'Lincoln_wave_2.png'와 같은 그림을 화면에 표시하는 프로그램을 작성하라.

참고 사이트: https://mymodernmet.com/tyler-foust-one-line-drawing-portraits/

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
import os
from PyQt5 import QtWidgets, QtGui
import sys

def block_mean(image, block_size=(16, 16)):
    """
    Compute the mean of each block in the image.

    Parameters:
    - image: A numpy array of the image.
    - block_size: A tuple indicating the size of the block.

    Returns:
    - A new numpy array with the block mean image.
    """
    image = 255 - image #이미지 색상을 반전
    height, width = image.shape #이미지 세로와 가로 크기 가져오기
    v_blocks = height // block_size[0] #세로 블록 수 계산
    h_blocks = width // block_size[1] #가로 블록 수 계산

    #블록 크기 나머지 부분 처리
    height_margin = (height % block_size[0]) // 2
    width_margin = (width % block_size[1]) // 2
    
    #결과 이미지 크기
    out_height = height // block_size[0]
    out_width = width // block_size[1]

    block_mean_image = np.zeros((out_height, out_width))

    #각 블록에 대해 평균값 계산
    for i in range(height_margin, height-height_margin, block_size[0]):
        for j in range(width_margin, width-width_margin, block_size[1]):
            block_mean = image[i:i+block_size[0], j:j+block_size[1]].mean()
            block_mean_image[i // block_size[0], j // block_size[1]] = block_mean

    return block_mean_image

def create_wave_figure(image, block_size=(16, 16)):
    """
    Create a Matplotlib figure with the wave plot.

    Parameters:
    - image: Input image array.
    - block_size: Tuple indicating the size of the block.

    Returns:
    - Matplotlib figure object.
    """
    dot_per_block = block_size[0]
    blk_mean_image = block_mean(image, block_size)
    print(blk_mean_image.shape)
    
    #그림 크기 설정
    fig = plt.Figure(figsize=(blk_mean_image.shape[1]*0.4, blk_mean_image.shape[0]*0.4))
    ax = fig.add_subplot(111)

    x = np.linspace(0, 2*np.pi*blk_mean_image.shape[1], dot_per_block * blk_mean_image.shape[1] + 1)
    for row in range(blk_mean_image.shape[0]): #각 행에 대해 사인 파형을 그리기
        y = np.sin(x)
        w = np.array([[y_val]*dot_per_block for y_val in blk_mean_image[row]]).flatten() / 255.
        ax.plot(x[:-1], w*y[:-1] - 2*row, 'k') #파형 그리기
    ax.set_xticks([]) #x축 눈금 제거
    ax.set_yticks([]) #y축 눈금 제거
    
    return fig

class WaveWindow(QtWidgets.QMainWindow):
    """
    PyQt5 window to display the wave plot.
    """
    def __init__(self, image, block_size=(16, 16), icon_path=None):
        super().__init__()
        self.setWindowTitle("Wave Image Viewer")

        
        if icon_path and os.path.exists(icon_path):
            self.setWindowIcon(QtGui.QIcon(icon_path)) #아이콘 설정
        else:
            print(f"Warning: Icon file '{icon_path}' not found or invalid. Using default icon.")

        self.figure = create_wave_figure(image, block_size)
        self.canvas = FigureCanvas(self.figure)

       
        self.setCentralWidget(self.canvas)

       
        self.resize(int(self.figure.get_size_inches()[0] * self.figure.dpi),
                    int(self.figure.get_size_inches()[1] * self.figure.dpi))

def show_wave_image(image, block_size=(16, 16), icon_path=None):
    """
    Display the wave plot in a PyQt5 window.

    Parameters:
    - image: Input image array.
    - block_size: Tuple indicating the size of the block.
    - icon_path: Path to the icon file (.ico, .png, etc.).
    """
    app = QtWidgets.QApplication(sys.argv)
    window = WaveWindow(image, block_size, icon_path)
    window.show()
    try:
        sys.exit(app.exec_())
    except SystemExit:
        pass


file_directory = './'
file_name = 'Lincoln.jpg'
icon_path = 'mspaint.png' #아이콘 바꾸기 
BLK_SIZE = (32, 32)
file_path = os.path.join(file_directory, file_name)


image = plt.imread(file_path)


show_wave_image(image, BLK_SIZE, icon_path)

(33, 26)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
import os
from PyQt5 import QtWidgets, QtGui
import sys

def block_mean(image, block_size=(16, 16)):
    """
    Compute the mean of each block in the image.

    Parameters:
    - image: A numpy array of the image.
    - block_size: A tuple indicating the size of the block.

    Returns:
    - A new numpy array with the block mean image.
    """
    image = 255 - image
    height, width = image.shape
    v_blocks = height // block_size[0]
    h_blocks = width // block_size[1]

    height_margin = (height % block_size[0]) // 2
    width_margin = (width % block_size[1]) // 2
    
    out_height = height // block_size[0]
    out_width = width // block_size[1]

    block_mean_image = np.zeros((out_height, out_width))

    for i in range(height_margin, height-height_margin, block_size[0]):
        for j in range(width_margin, width-width_margin, block_size[1]):
            block_mean = image[i:i+block_size[0], j:j+block_size[1]].mean()
            block_mean_image[i // block_size[0], j // block_size[1]] = block_mean

    return block_mean_image

def create_wave_figure(image, block_size=(16, 16), randomness=30):
    """
    Create a Matplotlib figure with the wave plot in XKCD style.

    Parameters:
    - image: Input image array.
    - block_size: Tuple indicating the size of the block.
    - randomness: Randomness parameter for XKCD style.

    Returns:
    - Matplotlib figure object.
    """
    dot_per_block = block_size[0]
    blk_mean_image = block_mean(image, block_size)
    print(blk_mean_image.shape)
    
    fig = plt.Figure(figsize=(blk_mean_image.shape[1]*0.4, blk_mean_image.shape[0]*0.4))
    ax = fig.add_subplot(111)

    with plt.xkcd(randomness=randomness):
        x = np.linspace(0, 2*np.pi*blk_mean_image.shape[1], dot_per_block * blk_mean_image.shape[1] + 1)
        for row in range(blk_mean_image.shape[0]):
            y = np.sin(x)
            w = np.array([[y_val]*dot_per_block for y_val in blk_mean_image[row]]).flatten() / 255.
            ax.plot(x[:-1], w*y[:-1] - 2*row, 'k')
        ax.set_xticks([])
        ax.set_yticks([])
    
    return fig

class WaveWindow(QtWidgets.QMainWindow):
    """
    PyQt5 window to display the wave plot.
    """
    def __init__(self, image, block_size=(16, 16), randomness=30, icon_path=None):
        super().__init__()
        self.setWindowTitle("Wave Image Viewer")

        
        if icon_path and os.path.exists(icon_path):
            self.setWindowIcon(QtGui.QIcon(icon_path))
        else:
            print(f"Warning: Icon file '{icon_path}' not found or invalid. Using default icon.")

        
        self.figure = create_wave_figure(image, block_size, randomness)
        self.canvas = FigureCanvas(self.figure)

        
        self.setCentralWidget(self.canvas)

        
        self.resize(int(self.figure.get_size_inches()[0] * self.figure.dpi),
                    int(self.figure.get_size_inches()[1] * self.figure.dpi))

def show_wave_image(image, block_size=(16, 16), randomness=30, icon_path=None):
    """
    Display the wave plot in a PyQt5 window.

    Parameters:
    - image: Input image array.
    - block_size: Tuple indicating the size of the block.
    - randomness: Randomness parameter for XKCD style.
    - icon_path: Path to the icon file (.ico, .png, etc.).
    """
    app = QtWidgets.QApplication(sys.argv)
    window = WaveWindow(image, block_size, randomness, icon_path)
    window.show()
    try:
        sys.exit(app.exec_())
    except:
        pass

# File settings
file_directory = './'
file_name = 'Lincoln.jpg'
icon_path = 'mspaint.png'  
BLK_SIZE = (8, 8)
RANDOMNESS = 10
file_path = os.path.join(file_directory, file_name)


try:
    image = plt.imread(file_path)
except FileNotFoundError:
    print(f"Error: Image file '{file_path}' not found.")
    sys.exit(1)


show_wave_image(image, BLK_SIZE, RANDOMNESS, icon_path)

(135, 104)
